In [ ]:
import * as tslab from "tslab";
import { readFileSync } from "fs";

const css = readFileSync("../style.css", "utf-8");
tslab.display.html(`<style>${css}</style>`);

# <a href="https://en.wikipedia.org/wiki/Unification_(computer_science)">Unification</a>

This notebook implements the algorithm of *Martelli and Montanari* for the unification of terms.

## Utility Functions

In [ ]:
import { LogicParser, Formula } from './FOL-Parser';

Formulas are represented as nested arrays. In order to convert a string into a nested array we use the class `LogicParser` that is implemented in the module `FOL-Parser`. Our parser distinguishes variables and function symbols as follows:
- A word starting with a lower case letter is interpreted as a *variable*.
- A word starting with an upper case letter is assumed to be a *function* or *predicate symbol*.

In [ ]:
function parseTerm(s: string): Formula {
    const parser = new LogicParser(s);
    return parser.parse();
}

In [ ]:
parseTerm('F(G(x),y)');

The method $\texttt{apply}(t, \sigma)$ takes an object $t$ and a substitution $\sigma$ and computes $t\sigma$, i.e. it *applies* the substitution $\sigma$ to $t$. The object $t$ is either a term, a *syntactic equation*, or a `Set` of syntactic equations. The substitution $\sigma$ is represented as a JavaScript object (Record). Assume that $\sigma = \bigl\{ x_1 \mapsto t_1, \cdots, x_n \mapsto t_n \bigr\}$. Then $t\sigma$ is defined by induction on $t$:
- If $t$ is a variable, there are two cases when defining $t\sigma$:
  - $t = x_i$ for an $i\in\{1,\cdots,n\}$. Then we define 
    $$ x_i\sigma := t_i. $$
  - $t = y$ where $y\in\mathcal{V}$, but $y \not\in \{x_1,\cdots,x_n\}$. Then we define 
    $$ y\sigma := y.$$
- Otherwise, we must have $t = f(s_1,\cdots,s_m)$. Then we define: 
  $$ f(s_1, \cdots, s_m)\sigma := f(s_1\sigma, \cdots, s_m\sigma). $$

In [ ]:
function apply(t: Formula | Set<Formula>, σ: Record<string, Formula>): Formula | Set<Formula> {
    "Apply the substitution σ to the term t."
    if (t instanceof Set) { // t is a set of equations or a clause
        return new Set([...t].map(e => apply(e as Formula, σ) as Formula));
    }
    if (typeof t === 'string') { // t is a variable
        if (t in σ) {
            return σ[t];
        } else {
            return t;
        }
    } else { 
        // at this point, t must be a term, a literal, or a syntactic equation (Array)
        const [f, ...Ts] = t;
        return [f, ...Ts.map(s => apply(s, σ) as Formula)];
    }
}

In [ ]:
const s1 = parseTerm('G(z)');
const s2 = parseTerm('H(u, v)');
const σ: Record<string, Formula> = { 'x': s1, 'y': s2 };
σ

In [ ]:
const t = parseTerm('F(x,H(y,x),G(z))')
t

In [ ]:
apply(t, σ);

If  $\sigma = \big\{ x_1 \mapsto s_1, \cdots, x_m \mapsto s_m \big\}$ and
$\tau = \big\{ y_1 \mapsto t_1, \cdots, y_n \mapsto t_n \big\}$ 
are two substitutions that are <em style="color:blue;">non-overlapping</em>, i.e. such that $\texttt{dom}(\sigma) \cap \texttt{dom}(\tau) = \{\}$ holds,
then we define the <em style="color:blue;">composition</em> $\sigma\tau$ of $\sigma$ and $\tau$ as follows:
$$\sigma\tau := \big\{ x_1 \mapsto s_1\tau, \cdots, x_m \mapsto s_m\tau,\; y_1 \mapsto t_1, \cdots, y_n \mapsto t_n \big\}$$
The function $\texttt{compose}(\sigma, \tau)$ takes two non-overlapping substitutions and computes the composition $\sigma\tau$.

In [ ]:
function compose(σ: Record<string, Formula>, τ: Record<string, Formula>): Record<string, Formula> {
    const result: Record<string, Formula> = { ...τ };
    for (const [x, s] of Object.entries(σ)) {
        result[x] = apply(s, τ) as Formula;
    }
    return result;
}

In [ ]:
const τ = { 'z': s1, 'u': s2 };
τ

In [ ]:
σ

In [ ]:
compose(σ, τ);

The function $\texttt{occurs}(x, t)$ checks whether the variable $x$ occurs in the term $t$.

In [ ]:
function occurs(x: string, t: Formula): boolean {
    if (x === t) {
        return true;
    }
    if (typeof t === 'string') { // t must be a variable different from x
        return false;
    }
    // t is [f, args...]
    const args = t.slice(1) as Formula[];
    return args.some(arg => occurs(x, arg));
}

In [ ]:
t

In [ ]:
occurs('u', t);

In [ ]:
occurs('x', t);

## The Algorithm of Martelli and Montanari

The rules of Martelli and Montanari that can be used to solve a system of syntactical equations are as follows:
<ol>
<li> If $y\in\mathcal{V}$ is a variable that does <b style="color:red;">not</b> occur in the term $t$,
     then we perform the following reduction: 
     $$ \Big\langle E \cup \big\{ y \doteq t \big\}, \sigma \Big\rangle \quad\leadsto \quad 
         \Big\langle E\{y \mapsto t\}, \sigma\big\{ y \mapsto t \big\} \Big\rangle 
     $$
</li>      
<li> If the variable $y$ occurs in the term $t$, then the system of syntactical equations
     $E \cup \big\{ y \doteq t \big\}$ is not solvable:
     $$ \Big\langle E \cup \big\{ y \doteq t \big\}, \sigma \Big\rangle\;\leadsto\; \texttt{None} \quad
        \mbox{if $y \in \textrm{Var}(t)$ and $y \not=t$.}$$
</li>
<li> If $y\in\mathcal{V}$ is a variable and $t$ is no variable, then we use the following rule:
     $$ \Big\langle E \cup \big\{ t \doteq y \big\}, \sigma \Big\rangle \quad\leadsto \quad 
         \Big\langle E \cup \big\{ y \doteq t \big\}, \sigma \Big\rangle.
     $$   
</li>
<li> Trivial syntactical equations of variables can be dropped:
     $$ \Big\langle E \cup \big\{ x \doteq x \big\}, \sigma \Big\rangle \quad\leadsto \quad
         \Big\langle E, \sigma \Big\rangle.
     $$   
</li>
<li> If $f$ is an $n$-ary function symbol, then we have: 
     $$ \Big\langle E \cup \big\{ f(s_1,\cdots,s_n) \doteq f(t_1,\cdots,t_n) \big\}, \sigma \Big\rangle 
         \;\leadsto\; 
         \Big\langle E \cup \big\{ s_1 \doteq t_1, \cdots, s_n \doteq t_n\}, \sigma \Big\rangle.
     $$   
</li>
<li> The system of syntactical equations $E \cup \big\{ f(s_1,\cdots,s_m) \doteq g(t_1,\cdots,t_n) \big\}$
     has <b style="color:red;">no</b> solution if the function symbols $f$ and $g$ are different:
     $$ \Big\langle E \cup \big\{ f(s_1,\cdots,s_m) \doteq g(t_1,\cdots,t_n) \big\},
      \sigma \Big\rangle \;\leadsto\; \texttt{None} \qquad \mbox{if $f \not= g$}.
     $$
</ol>


Given a set of <em style="color:blue;">syntactical equations</em> $E$ and a substitution $\sigma$, the function $\texttt{solve}(E, \sigma)$ uses the rules of Martelli and Montanari to solve $E$.

In [ ]:
function solve(E: Set<Formula>, σ: Record<string, Formula>): Record<string, Formula> | null {
    while (E.size > 0) {
        const equation = E.values().next().value;
        E.delete(equation);
        const [_, s, t] = equation as [string, Formula, Formula];
        // remove trivial equations
        if (JSON.stringify(s) === JSON.stringify(t)) {
            continue;
        }

        if (typeof s === 'string') { // s is a variable
            if (occurs(s, t)) {
                return null;
            } else {
                const newE = apply(E, { [s]: t }) as Set<Formula>;
                E = newE; 
                
                σ = compose(σ, { [s]: t });
            }
        } else if (typeof t === 'string') { // t is a variable, but s is not
            E.add(['≐', t, s]);
        } else {
            const [f, ...sArgs] = s as [string, ...Formula[]];
            const [g, ...tArgs] = t as [string, ...Formula[]];
            const m = sArgs.length;
            const n = tArgs.length;

            if (f !== g || m !== n) {
                return null;
            } else {
                for (let i = 0; i < m; i++) {
                    E.add(['≐', sArgs[i], tArgs[i]]);
                }
            }
        }
    }
    return σ;
}

Given two terms $s$ and $t$, the function $\texttt{unify}(s, t)$ computes the <em style="color:blue;">most general unifier</em> of $s$ and $t$.

In [ ]:
function unify(s: Formula, t: Formula): Record<string, Formula> | null {
    return solve(new Set([['≐', s, t]]), {});
}

In [ ]:
const t1 = parseTerm('P(x1,F(x1))');
const t2 = parseTerm('P(G(x2),x3)');
[t1, t2];

In [ ]:
const μ = unify(t1, t2);
μ;